# LoRA-Edit: Interactive Modal Session

Welcome to the interactive notebook for LoRA-Edit, running on Modal!

This environment is pre-configured with all dependencies and models downloaded. The `/data` directory is a persistent network file system where you can store your work.

**Workflow:**
1.  **Upload Data (if you haven't already):** In your *local* terminal, upload your preprocessed data to Modal's NFS: `modal nfs put lora-edit-data processed_data/my_awesome_video /my_awesome_video`
2.  **Run Preprocessing (Alternative):** Run the Gradio UI in the first cell to preprocess data directly within this cloud environment.
3.  **Train:** Run the training cell.
4.  **Inference:** Run the inference cells.

## 1. Preprocessing via Interactive UI

This cell launches the Gradio UI. A public link will be printed. Open it to preprocess your video.

**Instructions:**
- In the UI, set the **Model Checkpoint Path** to `/root/models/Wan2.1-I2V-14B-480P`.
- Set the **Data Processing Save Path** to a path inside `/data`, for example: `/data/my_awesome_video`.
- After you are done, stop this cell's execution.

In [ ]:
# The predata_app.py script was modified to enable sharing.
!python predata_app.py --checkpoint_dir /root/models/models_sam/sam2_hiera_large.pt

## 2. LoRA Training

This cell runs the training process using the data you just preprocessed.

In [ ]:
import os

# IMPORTANT: Change this to match the sequence name you used in the UI.
sequence_name = 'my_awesome_video'
training_config_path = f'/data/{sequence_name}/configs/training.toml'

if not os.path.exists(training_config_path):
    print(f'❌ Error: Training config not found at {training_config_path}')
else:
    print(f'Starting LoRA training for {sequence_name}...')
    !NCCL_P2P_DISABLE="1" NCCL_IB_DISABLE="1" deepspeed --num_gpus=1 train.py --deepspeed --config {training_config_path}

## 3. Inference

First, upload your edited first frame. Then, run the inference script.

In [ ]:
from google.colab import files # This is a misnomer, it works in many Jupyter environments
import os
from PIL import Image

# IMPORTANT: Change this to match your sequence name.
sequence_name = 'my_awesome_video'
data_dir = f'/data/{sequence_name}'

print('Please upload your edited first frame.')
uploaded_edit = files.upload()

if uploaded_edit:
    edited_filename = list(uploaded_edit.keys())[0]
    output_path = os.path.join(data_dir, 'edited_image.png')
    Image.open(edited_filename).save(output_path)
    os.remove(edited_filename)
    print(f"✅ Edited frame saved to '{output_path}'")

In [ ]:
import os

sequence_name = 'my_awesome_video'
data_dir = f'/data/{sequence_name}'
wan_model_path = '/root/models/Wan2.1-I2V-14B-480P'

print('Starting inference...')
!python inference.py --model_root_dir {wan_model_path} --data_dir {data_dir}

### View the result!

In [ ]:
from IPython.display import HTML
from base64 import b64encode
import os

sequence_name = 'my_awesome_video'
video_path = f'/data/{sequence_name}/edited_video.mp4'

if os.path.exists(video_path):
    mp4 = open(video_path,'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4).decode()
    display(HTML(f'<video width=400 controls><source src="{data_url}" type="video/mp4"></video>'))
else:
    print(f'❌ Video not found at {video_path}')